In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

In [2]:
X = tf.range(10)

In [3]:
X

<tf.Tensor: shape=(10,), dtype=int32, numpy=array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32)>

In [4]:
dataset = tf.data.Dataset.from_tensor_slices(X)

In [5]:
for item in dataset:
    print(item)

tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(2, shape=(), dtype=int32)
tf.Tensor(3, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)
tf.Tensor(5, shape=(), dtype=int32)
tf.Tensor(6, shape=(), dtype=int32)
tf.Tensor(7, shape=(), dtype=int32)
tf.Tensor(8, shape=(), dtype=int32)
tf.Tensor(9, shape=(), dtype=int32)


In [6]:
dataset = dataset.repeat(3).batch(7)
for item in dataset:
    print(item)

tf.Tensor([0 1 2 3 4 5 6], shape=(7,), dtype=int32)
tf.Tensor([7 8 9 0 1 2 3], shape=(7,), dtype=int32)
tf.Tensor([4 5 6 7 8 9 0], shape=(7,), dtype=int32)
tf.Tensor([1 2 3 4 5 6 7], shape=(7,), dtype=int32)
tf.Tensor([8 9], shape=(2,), dtype=int32)


In [7]:
dataset = dataset.map(lambda x:x*2)
for i in dataset:
    print(i)

tf.Tensor([ 0  2  4  6  8 10 12], shape=(7,), dtype=int32)
tf.Tensor([14 16 18  0  2  4  6], shape=(7,), dtype=int32)
tf.Tensor([ 8 10 12 14 16 18  0], shape=(7,), dtype=int32)
tf.Tensor([ 2  4  6  8 10 12 14], shape=(7,), dtype=int32)
tf.Tensor([16 18], shape=(2,), dtype=int32)


In [8]:
dataset = dataset.apply(tf.data.experimental.unbatch())

Instructions for updating:
Use `tf.data.Dataset.unbatch()`.


In [9]:
new_dataset = tf.range(10)

In [10]:
new_dataset = tf.data.Dataset.from_tensor_slices(new_dataset)

In [11]:
new_dataset = new_dataset.filter(lambda x:x<5)

In [12]:
for i in new_dataset:
    print(i)

tf.Tensor(0, shape=(), dtype=int32)
tf.Tensor(1, shape=(), dtype=int32)
tf.Tensor(2, shape=(), dtype=int32)
tf.Tensor(3, shape=(), dtype=int32)
tf.Tensor(4, shape=(), dtype=int32)


In [13]:
for i in new_dataset.take(1):
    print(i)

tf.Tensor(0, shape=(), dtype=int32)


In [14]:
n_ds = tf.data.Dataset.range(10).repeat(3)

In [15]:
n_ds = n_ds.shuffle(buffer_size=5, seed=42).batch(7)

In [16]:
for i in n_ds:
    print(i)

tf.Tensor([0 2 3 6 7 9 4], shape=(7,), dtype=int64)
tf.Tensor([5 0 1 1 8 6 5], shape=(7,), dtype=int64)
tf.Tensor([4 8 7 1 2 3 0], shape=(7,), dtype=int64)
tf.Tensor([5 4 2 7 8 9 9], shape=(7,), dtype=int64)
tf.Tensor([3 6], shape=(2,), dtype=int64)


In [17]:
# filepath_ds = tf.data.Dataset.list_files(train_filepaths, seed=42)
# n_readers = 5 
# dataset = filepath_dataset.interleave(
#     lambda filepath: tf.data.TextLineDataset(filepath).skip(1),
#     cycle_length=n_readers)

In [18]:
# for line in dataset.take(5):
#     print(line.numpy())

In [19]:
# X_mean, X_std = []
# n_inputs = 8

# def preprocess(line):
#     defs = [0.] * n_inputs + [tf.constant([]. dtype=tf.float32)]
#     fields = tf.io.decode_csv(line, record_defaults=def)
#     x = tf.stack(fields[:-1])
#     y = tf.stack(fields[-1:])
#     return(x-X_mean) / X_std, y

In [20]:
def csv_reader_dataset(filepaths, repeat=1, n_readers=5,
                       n_read_threads=None, shuffle_buffer_size=10000,
                       n_parse_threads=5, batch_size=32):
    dataset = tf.data.Dataset.list_files(filepaths).repeat(repeat)
    dataset = dataset.interleave(
        lambda filepath: tf.data.TextLineDataset(filepath).skip(1),
        cycle_length=n_readers, num_parallel_calls=n_read_threads)
    dataset = dataset.shuffle(shuffle_buffer_size)
    dataset = dataset.map(preprocess, num_parallel_calls=n_parse_threads)
    dataset = dataset.batch(batch_size)
    return dataset.prefetch(1)

In [21]:
# train_set = csv_reader_dataset(train_filepaths)
# valid_set = csv_reader_dataset(valid_filepaths)
# test_set = csv_reader_dataset(test_filepaths)

#model = Sequential([..])
#model.compile()
#model.fit(train_set,epochs=100,validation_data=valid_set)

In [22]:
with tf.io.TFRecordWriter("my_data.tfrecord") as f:
    f.write(b"this is the first record")
    f.write(b"and this is the second record")

In [23]:
filepaths = ["my_data.tfrecord"]
dataset = tf.data.TFRecordDataset(filepaths)
for item in dataset:
    print(item)

tf.Tensor(b'this is the first record', shape=(), dtype=string)
tf.Tensor(b'and this is the second record', shape=(), dtype=string)


In [24]:
options = tf.io.TFRecordOptions(compression_type="GZIP")
with tf.io.TFRecordWriter("my_compressed.tfrecord",options) as f:
    f.write(b"this is the compressed record")
    f.write(b"and this is the compressed second record")

In [25]:
dataset = tf.data.TFRecordDataset(["my_compressed.tfrecord"],
                                  compression_type="GZIP")

In [26]:
# syntax = "proto3";
# message Person{
#     string name = 1;
#     int32 id =2;
#     repeated string email = 3;}

In [31]:
# from person_pb2 import Person

# person = Person(name="AL", id=123, email=["a@b.com"])

In [32]:
from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Feature, Features, Example

In [34]:
person_example = Example(
    features = Features(
        feature={
            "name": Feature(bytes_list = BytesList(value=[b"Alice"])),
            "id": Feature(int64_list = Int64List(value=[123])),
            "emails": Feature(bytes_list = BytesList(value=[b"a@b.com", b"c@d.com"]))
        }))

In [35]:
with tf.io.TFRecordWriter("my_contacts.tfrecord") as f:
    f.write(person_example.SerializeToString())

In [38]:
feature_description = {
    "name" : tf.io.FixedLenFeature([], tf.string, default_value=""),
    "id": tf.io.FixedLenFeature([], tf.int64, default_value=0),
    "emails":tf.io.VarLenFeature(tf.string),
}

for serialised_example in tf.data.TFRecordDataset(["my_contacts.tfrecord"]):
    parsed_example = tf.io.parse_single_example(serialised_example, feature_description)

In [39]:
tf.sparse.to_dense(parsed_example["emails"], default_value=b"")

<tf.Tensor: shape=(2,), dtype=string, numpy=array([b'a@b.com', b'c@d.com'], dtype=object)>

In [40]:
parsed_example["emails"].values

<tf.Tensor: shape=(2,), dtype=string, numpy=array([b'a@b.com', b'c@d.com'], dtype=object)>

In [45]:
dataset = tf.data.TFRecordDataset(["my_contacts.tfrecord"]).batch(10)
for serialised_examples in dataset:
    parsed_examples = tf.io.parse_example(serialised_examples, feature_description)